#SILVER LAYER


In [0]:
from pyspark.sql.functions import *
# Define Active Storage Path
active_path = "abfss://active-data@strgarchiveproject01.dfs.core.windows.net/"

# READ BRONZE TABLE
bronze_df = spark.read.format("delta") \
    .load(active_path + "bronze_ecommerce")


In [0]:
bronze_df.select("InvoiceDate").show(10, False)

+---------------+
|InvoiceDate    |
+---------------+
|11/6/2011 16:13|
|11/6/2011 16:13|
|11/6/2011 16:13|
|11/6/2011 16:13|
|11/6/2011 16:13|
|11/6/2011 16:13|
|11/6/2011 16:13|
|11/6/2011 16:13|
|11/6/2011 16:13|
|11/6/2011 16:13|
+---------------+
only showing top 10 rows



In [0]:
# CLEAN DATA
silver_df = bronze_df.dropna(
    subset=["CustomerID", "InvoiceDate"]
)

In [0]:
silver_df = silver_df.withColumn(
    "InvoiceDateClean",
    trim(col("InvoiceDate").cast("string"))
)

In [0]:
# CONVERT TO TIMESTAMP
silver_df = silver_df.withColumn(
    "InvoiceDateConverted",
    unix_timestamp(
        col("InvoiceDateClean"),
        "M/d/yyyy H:mm"
    ).cast("timestamp")
)



In [0]:
# VERIFY CONVERSION
print("Converted Date Values")

silver_df.select(
    "InvoiceDate",
    "InvoiceDateClean",
    "InvoiceDateConverted"
).show(20, False)

Converted Date Values
+--------------+----------------+--------------------+
|InvoiceDate   |InvoiceDateClean|InvoiceDateConverted|
+--------------+----------------+--------------------+
|4/7/2011 12:16|4/7/2011 12:16  |2011-04-07 12:16:00 |
|4/7/2011 12:16|4/7/2011 12:16  |2011-04-07 12:16:00 |
|4/7/2011 12:16|4/7/2011 12:16  |2011-04-07 12:16:00 |
|4/7/2011 12:16|4/7/2011 12:16  |2011-04-07 12:16:00 |
|4/7/2011 12:16|4/7/2011 12:16  |2011-04-07 12:16:00 |
|4/7/2011 12:16|4/7/2011 12:16  |2011-04-07 12:16:00 |
|4/7/2011 12:16|4/7/2011 12:16  |2011-04-07 12:16:00 |
|4/7/2011 12:16|4/7/2011 12:16  |2011-04-07 12:16:00 |
|4/7/2011 12:16|4/7/2011 12:16  |2011-04-07 12:16:00 |
|4/7/2011 12:16|4/7/2011 12:16  |2011-04-07 12:16:00 |
|4/7/2011 12:16|4/7/2011 12:16  |2011-04-07 12:16:00 |
|4/7/2011 12:16|4/7/2011 12:16  |2011-04-07 12:16:00 |
|4/7/2011 12:16|4/7/2011 12:16  |2011-04-07 12:16:00 |
|4/7/2011 12:16|4/7/2011 12:16  |2011-04-07 12:16:00 |
|4/7/2011 12:16|4/7/2011 12:16  |2011-04-07

In [0]:
# DROP OLD DATE COLUMN
silver_df = silver_df.drop("InvoiceDate")
# RENAME CONVERTED COLUMN
silver_df = silver_df.withColumnRenamed(
    "InvoiceDateConverted",
    "InvoiceDate"
)

In [0]:
# DROP TEMP COLUMN
silver_df = silver_df.drop("InvoiceDateClean")

In [0]:
# ADD METADATA COLUMNS
silver_df = silver_df.withColumn(
    "ingestion_date",
    current_date()
)

silver_df = silver_df.withColumn(
    "record_status",
    lit("ACTIVE")
)

In [0]:
# FINAL PREVIEW
# ============================================

print("Final Silver Data")

silver_df.select(
    "InvoiceDate",
    "ingestion_date",
    "record_status"
).show(20, False)

display(silver_df)

Final Silver Data
+-------------------+--------------+-------------+
|InvoiceDate        |ingestion_date|record_status|
+-------------------+--------------+-------------+
|2011-11-06 16:13:00|2026-05-22    |ACTIVE       |
|2011-11-06 16:13:00|2026-05-22    |ACTIVE       |
|2011-11-06 16:13:00|2026-05-22    |ACTIVE       |
|2011-11-06 16:13:00|2026-05-22    |ACTIVE       |
|2011-11-06 16:13:00|2026-05-22    |ACTIVE       |
|2011-11-06 16:13:00|2026-05-22    |ACTIVE       |
|2011-11-06 16:13:00|2026-05-22    |ACTIVE       |
|2011-11-06 16:13:00|2026-05-22    |ACTIVE       |
|2011-11-06 16:13:00|2026-05-22    |ACTIVE       |
|2011-11-06 16:13:00|2026-05-22    |ACTIVE       |
|2011-11-06 16:13:00|2026-05-22    |ACTIVE       |
|2011-11-06 16:13:00|2026-05-22    |ACTIVE       |
|2011-11-06 16:13:00|2026-05-22    |ACTIVE       |
|2011-11-06 16:13:00|2026-05-22    |ACTIVE       |
|2011-11-06 16:13:00|2026-05-22    |ACTIVE       |
|2011-11-06 16:13:00|2026-05-22    |ACTIVE       |
|2011-11-06 1

InvoiceNo,StockCode,Description,Quantity,UnitPrice,CustomerID,Country,InvoiceDate,ingestion_date,record_status
574741,23426,METAL SIGN DROP YOUR PANTS,2,2.89,15993,United Kingdom,2011-11-06T16:13:00Z,2026-05-22,ACTIVE
574741,82567,"AIRLINE LOUNGE,METAL SIGN",1,2.1,15993,United Kingdom,2011-11-06T16:13:00Z,2026-05-22,ACTIVE
574741,22413,METAL SIGN TAKE IT OR LEAVE IT,1,2.95,15993,United Kingdom,2011-11-06T16:13:00Z,2026-05-22,ACTIVE
574741,84006,MAGIC TREE -PAPER FLOWERS,4,0.85,15993,United Kingdom,2011-11-06T16:13:00Z,2026-05-22,ACTIVE
574741,21787,RAIN PONCHO RETROSPOT,2,0.85,15993,United Kingdom,2011-11-06T16:13:00Z,2026-05-22,ACTIVE
574741,21786,POLKADOT RAIN HAT,2,0.42,15993,United Kingdom,2011-11-06T16:13:00Z,2026-05-22,ACTIVE
574741,22786,CUSHION COVER PINK UNION JACK,1,5.95,15993,United Kingdom,2011-11-06T16:13:00Z,2026-05-22,ACTIVE
574741,22423,REGENCY CAKESTAND 3 TIER,1,12.75,15993,United Kingdom,2011-11-06T16:13:00Z,2026-05-22,ACTIVE
574741,84978,HANGING HEART JAR T-LIGHT HOLDER,6,1.25,15993,United Kingdom,2011-11-06T16:13:00Z,2026-05-22,ACTIVE
574741,M,Manual,6,1.45,15993,United Kingdom,2011-11-06T16:13:00Z,2026-05-22,ACTIVE


In [0]:
# Save silver table
silver_df.write.format("delta") \
    .mode("overwrite") \
    .save(active_path + "silver_ecommerce")
print("Silver Layer Created Successfully")

Silver Layer Created Successfully
